# Laboratorio 05. Retropropagación

**Pregunta central:** ¿Cómo propaga una pérdida sus derivadas hasta todos los parámetros?

## Objetivos

- Derivar una neurona sigmoide con pérdida cuadrática.
- Comparar gradientes manuales con `Tensor.grad`.
- Hacer visible el orden `zero_grad`, `backward` y `step`.

## Prerrequisitos y conexión

Usa las capas y dimensiones del laboratorio 04; aquí empezamos a modificar los parámetros a partir del error.

## Teoría breve

La regla de la cadena compone derivadas locales desde la pérdida hasta cada parámetro: $\nabla_\theta L$. `backward()` acumula esas derivadas en los tensores hoja.

## Problema y datos

Derivamos primero una neurona de dos entradas y luego entrenamos XOR sobre su tabla exhaustiva, generada localmente.

## Evaluación

Comparamos derivadas manuales y automáticas con una tolerancia numérica, observamos normas de gradiente y reportamos exactitud sobre la tabla XOR. Esta exactitud comprueba el mecanismo, no la generalización.

### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F

SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)


**Resultado e interpretación.** La celda confirma el entorno utilizado. El resto del laboratorio está diseñado para CPU; una GPU disponible se utiliza sin ser requisito.


### Comparar una derivación manual con autograd

Para una neurona sigmoide y una pérdida cuadrática derivamos la cadena completa. El mismo valor se calcula con PyTorch antes de entrenar una red.

In [ ]:
x_one = torch.tensor([[0.4, -0.7]], dtype=torch.float64)
target_one = torch.tensor([[0.3]], dtype=torch.float64)
weight_one = torch.tensor([[0.2, -0.1]], dtype=torch.float64, requires_grad=True)
bias_one = torch.tensor([0.05], dtype=torch.float64, requires_grad=True)
probability_one = torch.sigmoid(x_one @ weight_one.T + bias_one)
loss_one = ((probability_one - target_one) ** 2).mean()
loss_one.backward()
manual_d_loss_d_probability = 2 * (probability_one.detach() - target_one)
manual_d_probability_d_logit = probability_one.detach() * (1 - probability_one.detach())
manual_d_logit = manual_d_loss_d_probability * manual_d_probability_d_logit
manual_weight_gradient = manual_d_logit.T @ x_one
manual_bias_gradient = manual_d_logit.sum(dim=0)
print('gradiente manual de pesos:', manual_weight_gradient.numpy().round(7))
print('gradiente autograd de pesos:', weight_one.grad.numpy().round(7))
print('gradiente manual del sesgo:', manual_bias_gradient.numpy().round(7))
print('gradiente autograd del sesgo:', bias_one.grad.numpy().round(7))
assert torch.allclose(manual_weight_gradient, weight_one.grad, atol=1e-12)
assert torch.allclose(manual_bias_gradient, bias_one.grad, atol=1e-12)

**Interpretación.** La coincidencia confirma la derivación de la regla de la cadena para este caso pequeño. Autograd evita escribir manualmente las derivadas de redes más profundas, pero no corrige un modelo o una pérdida mal especificados.

### Inspeccionar un gradiente
Hacemos un paso de forward y backward antes de actualizar parámetros.


In [ ]:
X = torch.tensor([[0.,0.], [0.,1.], [1.,0.], [1.,1.]], device=device)
y = torch.tensor([[0.], [1.], [1.], [0.]], device=device)
model = nn.Sequential(nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 1)).to(device)
loss_fn = nn.BCEWithLogitsLoss()
logits = model(X)
loss = loss_fn(logits, y)
loss.backward()
print('pérdida inicial:', loss.item())
print('norma gradiente primera capa:', model[0].weight.grad.norm().item())


**Resultado e interpretación.** Los gradientes se almacenan en `.grad`; si no se limpian antes de otro paso, PyTorch los acumula.


### Entrenar XOR
Ejecutamos `zero_grad`, `backward` y `step` en ese orden y evaluamos las cuatro combinaciones.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.04)
loss_history, gradient_norm_history = [], []
for epoch in range(1200):
    logits = model(X)
    loss = loss_fn(logits, y)
    optimizer.zero_grad()
    loss.backward()
    gradient_norm_history.append(model[0].weight.grad.norm().item())
    optimizer.step()
    loss_history.append(loss.item())
with torch.no_grad():
    probabilities = torch.sigmoid(model(X))
    predictions = (probabilities >= 0.5).float()
print('probabilidades:', probabilities.squeeze().cpu().numpy().round(2))
print('exactitud XOR:', (predictions == y).float().mean().item())
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(loss_history)
axes[0].set(xlabel='Época', ylabel='BCE', title='Pérdida')
axes[1].plot(gradient_norm_history)
axes[1].set(xlabel='Época', ylabel='Norma L2', title='Gradiente de la primera capa')
plt.tight_layout()
plt.show()

**Resultado e interpretación.** La curva y las predicciones permiten comprobar si la red aprendió esta tabla. Un conjunto de cuatro puntos solo ilustra el mecanismo, no garantiza robustez.


## Limitaciones y conclusiones

El gradiente manual se deriva para una neurona y una observación; la red XOR amplía el ejemplo, pero sus cuatro casos no permiten estimar rendimiento futuro. Los gradientes pueden ser pequeños, grandes o inestables según activación, inicialización y profundidad.

## Ejercicios

1. **Guiado:** cambia el objetivo `target_one` y verifica que ambos gradientes manuales y automáticos cambien en la misma dirección.
2. **Independiente:** dibuja la norma del gradiente de la primera capa durante el entrenamiento y relaciónala con la curva de pérdida.
3. **Desafío:** compara gradientes de una red con dos y seis capas ocultas usando la misma entrada y analiza si se atenúan o amplifican.

## Conexión con el siguiente laboratorio

El siguiente laboratorio contrasta funciones de pérdida y muestra cómo la elección del objetivo cambia la señal de aprendizaje.